# Plume: full pipeline on Kaggle

Clones [cryst4lpepsi/plume](https://github.com/cryst4lpepsi/plume), downloads the data, trains every model and re-scores everything.

**Before running**
- Session options → **Internet: On** (needs a phone-verified account).
- Session options → **Accelerator: None**. Training is CPU-only; a GPU session has fewer CPU cores and would be slower.
- Use **Save Version → Save & Run All (Commit)** so it runs on Kaggle's machines with your computer off. Expect roughly 6 to 10 hours. Kaggle stops a session at 12 hours; if that happens, run it again and it continues from the checkpoints.

**Checkpoints.** Progress is saved as it goes, so a crash, a timeout or a stopped session doesn't start everything over:
- *Cloud copy (recommended).* Add two secrets under Add-ons → Secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com → Settings → API → Create New Token, which downloads `kaggle.json` holding both). The notebook then keeps two **private** datasets, `plume-ckpt2-data` and `plume-ckpt2-progress`, updated during the run. The next run downloads them on its own and continues. Checkpoints of the first published run (`plume-ckpt-data`, `plume-ckpt-progress`) hold different stations and features; they are ignored and can be deleted.
- *Without secrets.* The same checkpoints are written to `/kaggle/working/plume_state`. To resume from a failed version, attach its output as input (Add Input → Your Work → this notebook) before re-running.

**Fires.** For the fire features, add a third secret `FIRMS_MAP_KEY` (free, from https://firms.modaps.eosdis.nasa.gov/api/map_key/). Without it the run still works and those two features are zero.

What is kept: the data stage's downloads as they arrive, the finished data stage (processed data and tensors), every finished training run with its predictions and logs, and the final results. A training run that was cut off part-way re-runs from the start.

**What you get** in the Output tab: `plume_results.zip` (results, app data, refreshed README and REPORT tables, and the model bundles in `data/bundle/`) and `plume_state/` (the checkpoints).

In [ ]:
import glob, json, os, shutil, subprocess, threading, time, urllib.request

REPO_URL = "https://github.com/cryst4lpepsi/plume.git"
BRANCH = "main"
SCRATCH = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
REPO = f"{SCRATCH}/plume"
OUT = "/kaggle/working"
STATE = f"{OUT}/plume_state"            # saved with the notebook version
STAGE = f"{SCRATCH}/ckpt"                # where tars are built and uploaded from
LOCAL_SPARE_GB = 2                       # local checkpoint copies always leave this much of /kaggle/working free
GEN = "2"                                # checkpoint generation; raised when stations or features change

# What each checkpoint holds, relative to the repo root.
PARTS = {
    "data": ["data/processed", "data/tensors", "data/.done_data"],
    "progress": ["data/runs", "data/preds", "data/logs", "results/baselines.json"],
    # the data stage's downloads, saved while it runs: each download step skips files it already has
    "raw": ["data/geo", "data/discover", "data/raw_hourly", "data/raw_hourly_eea", "data/om", "data/firms"],
}


def sh(cmd, cwd=None, check=True):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0 and check:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")
    print(f"[done in {(time.time() - t0) / 60:.1f} min] {cmd}", flush=True)
    return p.returncode


try:
    urllib.request.urlopen("https://github.com", timeout=15)
except Exception:
    raise RuntimeError("No internet. In the right panel open Session options and switch Internet on "
                       "(needs a phone-verified Kaggle account), then run this cell again.") from None

# Optional cloud checkpoints through Kaggle Secrets.
KUSER = None
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
    KUSER = os.environ["KAGGLE_USERNAME"]
    print(f"Cloud checkpoints on: private datasets under {KUSER}/plume-ckpt{GEN}-*")
except Exception:
    print("Cloud checkpoints off (no KAGGLE_USERNAME / KAGGLE_KEY secrets). Checkpoints go to /kaggle/working only.")

# Optional key for the NASA FIRMS fire detections.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["FIRMS_MAP_KEY"] = UserSecretsClient().get_secret("FIRMS_MAP_KEY")
    print("Fire features on (FIRMS_MAP_KEY found).")
except Exception:
    print("Fire features off (no FIRMS_MAP_KEY secret); they will be zero.")

for path in (SCRATCH, OUT):
    print(f"{path}: {shutil.disk_usage(path).free / 1e9:.0f} GB free")
print("CPU cores:", os.cpu_count())
if shutil.disk_usage(SCRATCH).free < 35e9:
    print("WARNING: under 35 GB of scratch space; the data stage may run out of disk.")

## Checkpoint helpers
`save(kind)` packs a checkpoint into one tar (named `.ckpt`, because Kaggle unpacks uploaded `.tar` files), keeps a copy in `/kaggle/working/plume_state` (if it fits) and uploads it as a new version of the private dataset. `restore()` finds the newest checkpoint, packed or already unpacked, and puts it back into the repo. A failed upload only prints a warning; it never stops the run.

In [ ]:
_lock = threading.Lock()


def _kaggle(args):
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    return r.returncode == 0 and "error" not in out.lower(), out


def _upload(kind, folder):
    slug = f"{KUSER}/plume-ckpt{GEN}-{kind}"
    with open(f"{folder}/dataset-metadata.json", "w") as f:
        json.dump({"title": f"plume-ckpt{GEN}-{kind}", "id": slug, "licenses": [{"name": "CC0-1.0"}]}, f)
    ok, msg = _kaggle(["datasets", "version", "-p", folder, "-m", time.strftime("%Y-%m-%d %H:%M")])
    if not ok:  # first time: create it (private by default)
        ok, msg = _kaggle(["datasets", "create", "-p", folder])
    if ok:
        print("uploaded", slug, flush=True)
    else:
        print(f"WARNING: upload failed for {slug}: {msg[-300:]}", flush=True)


def _keep_local(kind, folder, gb):
    """Copy the checkpoint to /kaggle/working (saved with the notebook version) only if it fits and still
    leaves LOCAL_SPARE_GB for the results. The old copy of the same checkpoint is replaced, so it counts as free."""
    local = f"{STATE}/{kind}/{kind}{GEN}.ckpt"
    try:
        free = shutil.disk_usage(OUT).free + (os.path.getsize(local) if os.path.exists(local) else 0)
        if gb * 1e9 < free - LOCAL_SPARE_GB * 1e9:
            os.makedirs(f"{STATE}/{kind}", exist_ok=True)
            shutil.copy(f"{folder}/{kind}{GEN}.ckpt", local + ".tmp")
            os.replace(local + ".tmp", local)
            return
        print(f"{kind} checkpoint ({gb:.1f} GB) does not fit in /kaggle/working; cloud copy only.")
    except Exception as e:
        print(f"WARNING: local copy of {kind} failed: {e}", flush=True)
    for f in (local + ".tmp", local):   # a partial or out-of-date copy would only fill the disk or mislead a resume
        if os.path.exists(f):
            os.remove(f)


def save(kind):
    """Pack one checkpoint, keep a local copy if it fits, upload it; never raises. The upload does not
    depend on the local copy."""
    with _lock:
        try:
            parts = [p for p in PARTS[kind] if os.path.exists(f"{REPO}/{p}")]
            if not parts:
                return
            folder = f"{STAGE}/{kind}"
            os.makedirs(folder, exist_ok=True)
            tmp = f"{folder}/{kind}{GEN}.ckpt.tmp"
            # exit status 1 only means a file changed while being read (a run was writing it); the archive is
            # still usable and that file is picked up again by the next checkpoint
            r = subprocess.run(["tar", "--warning=no-file-changed", "-cf", tmp, "-C", REPO, *parts])
            if r.returncode not in (0, 1):
                raise RuntimeError(f"tar exited with {r.returncode}")
            os.replace(tmp, f"{folder}/{kind}{GEN}.ckpt")
            gb = os.path.getsize(f"{folder}/{kind}{GEN}.ckpt") / 1e9
            print(time.strftime("%H:%M:%S"), f"checkpoint {kind}: {gb:.2f} GB", flush=True)
        except Exception as e:
            print(f"WARNING: checkpoint {kind} failed: {e}", flush=True)
            return
        _keep_local(kind, folder, gb)
        if KUSER:
            try:
                _upload(kind, folder)
            except Exception as e:
                print(f"WARNING: upload of {kind} failed: {e}", flush=True)


def _find(kind):
    """Newest checkpoint: cloud dataset first, then any attached input. Returns a tar file (.ckpt, or .tar from
    older runs) or, for a dataset Kaggle unpacked on upload, the folder holding the unpacked files."""
    names = (f"{kind}{GEN}.ckpt", f"{kind}{GEN}.tar")
    if KUSER:
        d = f"{SCRATCH}/restore/{kind}"
        shutil.rmtree(d, ignore_errors=True)
        slug = f"{KUSER}/plume-ckpt{GEN}-{kind}"
        _kaggle(["datasets", "download", "-d", slug, "-p", d, "--unzip"])
        if any(os.path.exists(f"{d}/{p}") for p in PARTS[kind]):
            return d, "cloud (unpacked by Kaggle)"
        # a large dataset cannot be downloaded whole (the API answers 404), only file by file
        for n in names:
            if not glob.glob(f"{d}/**/{n}", recursive=True):
                _kaggle(["datasets", "download", "-d", slug, "-f", n, "-p", d, "--unzip"])
        hits = [h for n in names for h in glob.glob(f"{d}/**/{n}", recursive=True)]
        if hits:
            return hits[0], "cloud"
    hits = sorted((h for n in names for h in glob.glob(f"/kaggle/input/**/{n}", recursive=True)), key=os.path.getmtime)
    return (hits[-1], "input") if hits else (None, None)


def restore():
    for kind in PARTS:
        if kind == "raw" and os.path.exists(f"{REPO}/data/.done_data"):
            continue   # the finished data stage was restored; its downloads are not needed
        found, src = _find(kind)
        if not found:
            print(f"no {kind} checkpoint found; starting that part fresh")
            continue
        sh(f"cp -a {found}/. {REPO}/" if os.path.isdir(found) else f"tar -xf {found} -C {REPO}")
        print(f"restored {kind} checkpoint from {src}: {found}")
    # unpacking can drop the hidden marker; complete tensors for both regions mean the data stage finished
    if all(os.path.exists(f"{REPO}/data/tensors/{r}.npz") for r in ("europe", "asia")):
        open(f"{REPO}/data/.done_data", "a").close()


class Autosave:
    """Saves a checkpoint every `minutes` while a long cell runs, and once more at the end."""
    def __init__(self, kind, minutes=20):
        self.kind, self.every, self.stop = kind, minutes * 60, threading.Event()

    def _loop(self):
        while not self.stop.wait(self.every):
            save(self.kind)

    def __enter__(self):
        self.t = threading.Thread(target=self._loop, daemon=True)
        self.t.start()

    def __exit__(self, *exc):
        self.stop.set()
        self.t.join()
        save(self.kind)  # runs even if the cell failed

## 1. Clone, set up Python 3.13 with the pinned packages, restore checkpoints

In [ ]:
if not os.path.isdir(f"{REPO}/.git"):
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}", cwd=SCRATCH)
sh("git log --oneline -1")
sh("pip install -q uv")
sh("[ -x .venv/bin/python ] || uv venv --python 3.13 .venv")
sh("uv pip install -q --python .venv/bin/python torch --index-url https://download.pytorch.org/whl/cpu")
sh("uv pip install -q --python .venv/bin/python -r ml/requirements.txt")
sh(".venv/bin/python -c 'import torch, lightgbm, polars; print(\"torch\", torch.__version__)'")
restore()
# The repo ships results/baselines.json, which would make the queue skip the baselines and
# leave LightGBM out. Remove it unless the LightGBM predictions are already here.
sh("[ -e data/preds/europe_wtest_lightgbm.npy ] || rm -f results/baselines.json")

## 2. Data (about 1 hour, network-bound)
OpenAQ and EEA observations, station selection, ERA5 weather, CAMS and fires, then the model tensors. Skipped if a finished data checkpoint was restored. The downloads are checkpointed every 15 minutes while this runs, so after a crash the next run fetches only what was missing.

In [ ]:
if os.path.exists(f"{REPO}/data/.done_data"):
    print("Data stage already finished (restored from checkpoint); skipping.")
else:
    with Autosave("raw", minutes=15):   # a crash here only repeats the downloads not yet saved
        sh("make data")
    sh("touch data/.done_data")
    save("data")
sh("du -sh data/* | sort -h")

## 3. Train (roughly 5 to 8 hours on 4 cores)
Baselines and a few short tuning runs first, then every graph-model run with the setting that did best on validation, four at a time. Finished runs are skipped, including ones restored from a checkpoint. Progress is checkpointed every 20 minutes and when the cell ends, even on failure. Per-run logs are in `data/logs/`.

In [ ]:
try:
    with Autosave("progress", minutes=20):
        sh("make train")
finally:
    sh("tail -n 3 data/logs/*.log | tail -n 80", check=False)
sh("ls data/runs/*.json | wc -l")

## 4. Evaluate, export the app data, verify it, refresh the tables

In [ ]:
sh("make eval")
sh("git status --short | head -40")

## 5. Save the outputs

In [ ]:
sh("make bundle", check=False)   # model + inputs for one-station forecasts; optional
sh(f"rm -f {OUT}/plume_results.zip && zip -qr {OUT}/plume_results.zip results web/public/data README.md REPORT.md"
   " $(ls -d data/bundle 2>/dev/null)")
PARTS["progress"] += ["results", "web/public/data", "README.md", "REPORT.md"]
save("progress")
sh(f"du -sh {OUT}/* && git rev-parse --short HEAD", check=False)

## Afterwards, on your computer
Download `plume_results.zip` from the Output tab, unzip it over the repository root, then commit:

```bash
git add results web/public/data README.md REPORT.md
git commit -m "Re-run pipeline: refreshed results and app data"
git push
```

When you no longer need them, delete the `plume-ckpt2-data`, `plume-ckpt2-progress` and `plume-ckpt2-raw` datasets from kaggle.com → Your Work → Datasets. Until then, a new run that finds them skips straight to whatever was left unfinished.